In [7]:
#lcoal
from database.var_check import name_check
from database.var_check import time_check
# standard
import sqlite3 
import os
# third
import pandas as pd
import matplotlib.pyplot
import matplotlib.dates as mdates
import requests 
from http import HTTPStatus

ModuleNotFoundError: No module named 'database'

In [4]:
import sys
import os

print("Current working directory:", os.getcwd())
print("Python module search paths:")
for p in sys.path:
    print(p)


Current working directory: /home/leon/projs/dark/testing
Python module search paths:
/usr/lib/python314.zip
/usr/lib/python3.14
/usr/lib/python3.14/lib-dynload

/home/leon/projs/dark/dark.venv/lib/python3.14/site-packages


In [ ]:
# Creating a function for market

name = 'Gold Coin Chest'

start = "2025-11-16T1:00:00Z" #UTC Time
end = "2025-11-16T23:00:00Z"

has_sold = "1"

def url_builder():
    name_check(name)
    time_check(start, end)
    return f"https://api.darkerdb.com/v1/market?key={os.getenv("dark_api_key")}&item={name}&from={start}&to={end}&has_sold={has_sold}&condense=true&limit={limit}&page={page}"
url = url_builder()

with requests.Session() as ses:   
    limit = '50'
    page = '1'
    code = ses.get(url).status_code
    
    if code == 200:
        list = []
        byte_counter = 0
        json = {"pagination": {"count": int(limit)}}

        while json['pagination']['count'] != 0:
            req = ses.get(url)
            json = req.json()
            list.extend(json['body'])
            page = str(int(page) + 1)
            byte_counter += len(req.content) # 8 bits in a byte

        df = pd.json_normalize(list, sep=',')
        print(f'Status Code {code}: {HTTPStatus(code).phrase} | {HTTPStatus(code).description}' 
              f'\n get-requests = {page}'
              f'\n bytes = {byte_counter}')
    
    else: print(f"Status Code {code}: {HTTPStatus(code).phrase} | {HTTPStatus(code).description}" )

connection to sql successful
Status Code 200: OK | Request fulfilled, document follows
 get-requests = 2
 bytes = 471


In [ ]:
df

In [ ]:
conn = sqlite3.connect("Market.db")
cursor = conn.cursor()

In [ ]:
cursor.execute(f'DROP TABLE IF EXISTS "{item}"')

In [ ]:
# to_sql doesnt support priumary key
# SQLite doesnt allow contraint to be altered in

query = f'''
CREATE TABLE IF NOT EXISTS "{item}" (
    id INTEGER INTEGER,
    cursor INTEGER PRIMARY KEY,
    item_id TEXT,
    item TEXT,
    archetype TEXT,
    rarity TEXT,
    price INTEGER,
    price_per_unit TEXT,
    quantity INTEGER,
    created_at TEXT,
    expires_at TEXT,
    sold_at TEXT,
    has_sold INTEGER,
    has_expired INTEGER,
    seller TEXT
    )'''

cursor.execute(query)

In [ ]:
table.to_sql(item, conn, if_exists='append', index=False, method=None)

IntegrityError: UNIQUE constraint failed: Gold Coin Chest.cursor